# Fake News Detection Using NLP (BERT)
**Dataset:** Fake and Real News Dataset (Kaggle)
**Goal:** Classify news articles as Fake or Real
**Technique:** BERT (Bidirectional Encoder Representations from Transformers)
**Model:** Fine-tuned bert-base-uncased

## Dataset Source
https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

The dataset have two files:
-  — fake news articles
-  — real news articles



## 1. Install & Import Libraries

In [ ]:
# install required libraries
!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


## 2. Load & Prepare Dataset

In [ ]:
# Load Fake and Real news CSVs
fake_df = pd.read_csv("Fake.csv")
real_df = pd.read_csv("True.csv")

# Add labels: 0 = Fake, 1 = Real
fake_df["label"] = 0
real_df["label"] = 1

print("Fake news samples:", len(fake_df))
print("Real news samples:", len(real_df))

fake_df.head(3)


In [ ]:
# Combine and shuffle
df = pd.concat([fake_df, real_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Total samples:", len(df))
print("Columns:", df.columns.tolist())
df.head()


## 3. Exploratory Data Analysis

In [ ]:
# Label distribution
label_counts = df["label"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(["Fake (0)", "Real (1)"], label_counts.values, color=["#e74c3c", "#2ecc71"], edgecolor="black")
plt.title("News Label Distribution")
plt.ylabel("Count")
for i, v in enumerate(label_counts.values):
    plt.text(i, v + 50, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Article length distribution
df["text_length"] = df["text"].astype(str).apply(len)

plt.figure(figsize=(10, 4))
for label, color, name in [(0, "#e74c3c", "Fake"), (1, "#2ecc71", "Real")]:
    subset = df[df["label"] == label]["text_length"]
    plt.hist(subset, bins=50, alpha=0.6, color=color, label=name, edgecolor="black")
plt.title("Article Text Length Distribution")
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.legend(title="Label")
plt.tight_layout()
plt.show()

print("Average text length:")
print(df.groupby("label")["text_length"].mean().rename({0: "Fake", 1: "Real"}))


In [ ]:
# Subject distribution
if "subject" in df.columns:
    plt.figure(figsize=(12, 4))
    subject_churn = pd.crosstab(df["subject"], df["label"])
    subject_churn.columns = ["Fake", "Real"]
    subject_churn.plot(kind="bar", color=["#e74c3c", "#2ecc71"], edgecolor="black", figsize=(12, 4), rot=30)
    plt.title("Article Count by Subject & Label")
    plt.ylabel("Count")
    plt.legend(title="Label")
    plt.tight_layout()
    plt.show()


## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)           # remove special chars & numbers
    text = re.sub(r"\s+", " ", text).strip()          # remove extra whitespace
    return text

# Combine title + text for richer input
df["combined"] = df["title"].astype(str) + " " + df["text"].astype(str)
df["combined"] = df["combined"].apply(clean_text)

# Truncate to 512 tokens worth (~2000 chars) for BERT efficiency
df["combined"] = df["combined"].apply(lambda x: x[:2000])

print("Sample cleaned text:")
print(df["combined"][0][:300])


## 5. Train-Test Split

In [ ]:
X = df["combined"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")


## 6. BERT Tokenizer & Dataset Class

In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print("Tokenizer loaded!")


In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
MAX_LEN = 128
BATCH_SIZE = 16

train_dataset = NewsDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = NewsDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


## 7. Load BERT Model

In [ ]:
# Load pre-trained BERT for binary classification
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model = model.to(device)
print("BERT model loaded and moved to:", device)


## 8. Fine-Tune BERT

In [ ]:
EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Training for {EPOCHS} epochs | Total steps: {total_steps}")


In [ ]:
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} complete | Avg Loss: {avg_loss:.4f}")

print("Training complete!")


In [ ]:
# Plot training loss
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS+1), train_losses, marker="o", color="#3498db", linewidth=2)
plt.title("BERT Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(range(1, EPOCHS+1))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("=== BERT Classification Report ===")
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(classification_report(all_labels, all_preds, target_names=["Fake", "Real"]))


## 11. Save the Model

In [ ]:
# Save fine-tuned BERT model and tokenizer
model.save_pretrained("fake_news_bert_model")
tokenizer.save_pretrained("fake_news_bert_model")
print("Model saved to ./fake_news_bert_model/")
